# PePath Demo Notebook

## Pediatric Pathology Slide Search Tool

This notebook demonstrates the end-to-end PePath pipeline using:
- **Virchow2** patch embedding model
- **SAMPLER (Percentile)** slide embedding method

### What you'll learn:
1. **Setup & Verification** - Verify your environment is ready
2. **End-to-End Pipeline** - Process a WSI from scratch
3. **Slide Search** - Find morphologically similar slides with visualization
4. **ROI Search** - Find similar tissue regions with heatmap overlays

---

## 1. Setup & Verification

First, let's verify that your environment is correctly set up.

In [ ]:
# Environment check
import sys
import os

print("PePath Environment Check")
print("=" * 50)
print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {os.getcwd()}")

# Check GPU
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected. CPU mode will be used (slower).")

In [ ]:
# Check required packages
required_packages = [
    ('openslide', 'openslide-python + openslide-bin'),
    ('timm', 'timm'),
    ('huggingface_hub', 'huggingface_hub'),
]

print("Checking required packages...")
for pkg, install_name in required_packages:
    try:
        __import__(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        print(f"  [MISSING] {pkg} - Install with: pip install {install_name}")

# Check HuggingFace authentication (required for Virchow2)
print("\nChecking HuggingFace authentication...")
try:
    from huggingface_hub import HfApi
    api = HfApi()
    user = api.whoami()
    print(f"  [OK] Logged in as: {user['name']}")
except Exception as e:
    print(f"  [WARNING] Not authenticated. Run: hf auth login")
    print(f"  Virchow2 is a gated model - you need to request access and authenticate.")

---
## 2. Configuration

Set up paths for the demo. We'll use the sample data in the `data/` folder.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial.distance import cdist

# Configuration
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

# Input files
SLIDE_PATH = DATA_DIR / "4261.svs"  # Sample WSI
REFERENCE_MATRIX_PATH = DATA_DIR / "final_slides_sampler_matrix.pkl"  # Slide embeddings database
METADATA_PATH = DATA_DIR / "final_scan_id_disease_map.csv"  # Disease labels
REFERENCE_PATCHES_DIR = DATA_DIR / "patch_embeddings"  # Patch embeddings for ROI search

# Heatmap visualization files
MOSAIC_METADATA_PATH = DATA_DIR / "mosaic_metadata.pkl"  # Mosaic cluster locations
THUMBNAILS_DIR = DATA_DIR / "thumbnails"  # Slide thumbnails for overlay
DIMENSIONS_CSV = DATA_DIR / "slide_dimensions.csv"  # Slide dimensions for coordinate mapping

# Output directories
MOSAIC_DIR = OUTPUT_DIR / "mosaic"
PATCH_EMB_DIR = OUTPUT_DIR / "patch_embeddings"
SLIDE_EMB_DIR = OUTPUT_DIR / "slide_embeddings"
SEARCH_RESULTS_DIR = OUTPUT_DIR / "search_results"
HEATMAP_DIR = OUTPUT_DIR / "heatmaps"

# Create output directories
for d in [MOSAIC_DIR, PATCH_EMB_DIR, SLIDE_EMB_DIR, SEARCH_RESULTS_DIR, HEATMAP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Slide ID (extracted from filename)
SLIDE_ID = SLIDE_PATH.stem

print(f"Slide ID: {SLIDE_ID}")
print(f"Slide path: {SLIDE_PATH}")
print(f"Slide exists: {SLIDE_PATH.exists()}")
print(f"\nHeatmap resources:")
print(f"  Mosaic metadata: {MOSAIC_METADATA_PATH.exists()}")
print(f"  Thumbnails dir: {THUMBNAILS_DIR.exists()}")
print(f"  Dimensions CSV: {DIMENSIONS_CSV.exists()}")

---
## 3. End-to-End Pipeline

The pipeline consists of 4 steps:
1. **Mosaic Generation** - Extract tissue patches from WSI
2. **Patch Embedding** - Extract Virchow2 embeddings for each patch
3. **Slide Embedding** - Aggregate patches into slide-level embedding (percentile method)
4. **Search** - Find similar slides/ROIs

### Step 1: Mosaic Generation

Extract tissue patches from the whole slide image.

In [ ]:
from pepath.preprocessing.mosaic import MosaicGenerator

# Initialize mosaic generator
mosaic_gen = MosaicGenerator(
    base_patchsize=512,
    num_clusters=100,  # Number of representative patches to select
    verbose=True
)

# Generate mosaic
mosaic_output = MOSAIC_DIR / f"{SLIDE_ID}"
mosaic_output.mkdir(exist_ok=True)

print(f"Processing slide: {SLIDE_PATH}")
mosaic = mosaic_gen.process_slide(
    slide_fn=str(SLIDE_PATH),
    output_dir=str(mosaic_output)
)

print(f"\nMosaic generated: {len(mosaic)} patches selected")
print(f"Sample patch location: {mosaic[0]['wsi_loc']}")

### Step 2: Patch Embedding Extraction

Extract Virchow2 embeddings for each patch.

In [ ]:
from pepath.embeddings.patch_embedding import Virchow2EmbeddingExtractor

# Initialize Virchow2 extractor
virchow2 = Virchow2EmbeddingExtractor(
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=64,
    verbose=True
)

# Extract embeddings
print(f"Extracting Virchow2 embeddings for {len(mosaic)} patches...")
patch_embeddings = virchow2.extract_slide_embeddings(
    slide_path=str(SLIDE_PATH),
    mosaic=mosaic
)

print(f"\nPatch embeddings shape: {patch_embeddings.shape}")
print(f"Embedding dimension: {patch_embeddings.shape[1]}")

# Save patch embeddings (in tmp subdirectory, matching pipeline structure)
patch_emb_subdir = PATCH_EMB_DIR / "tmp"
patch_emb_subdir.mkdir(exist_ok=True)
patch_emb_path = patch_emb_subdir / f"{SLIDE_ID}.pkl"
with open(patch_emb_path, 'wb') as f:
    pickle.dump((patch_embeddings, mosaic), f)
print(f"Saved to: {patch_emb_path}")

### Step 3: Slide Embedding Generation

Aggregate patch embeddings into a slide-level representation using the percentile method.

In [ ]:
from pepath.embeddings.slide_embedding import PercentileSlideEmbeddingGenerator

# Initialize percentile embedding generator
percentile_gen = PercentileSlideEmbeddingGenerator(
    feature_dim=1280,  # Virchow2 class token dimension
    verbose=True
)

# Generate slide embedding
slide_embedding = percentile_gen.process_single_slide(patch_embeddings)

print(f"Slide embedding shape: {slide_embedding.shape}")
print(f"  - 10 percentiles x 1280 features = {10 * 1280} dimensions (flattened)")

# Save slide embedding
slide_emb_path = SLIDE_EMB_DIR / f"{SLIDE_ID}.pkl"
with open(slide_emb_path, 'wb') as f:
    pickle.dump(slide_embedding, f)
print(f"Saved to: {slide_emb_path}")

---
## 4. Slide Search

Find morphologically similar slides from the reference database.

In [ ]:
# Load reference database
print("Loading reference database...")
with open(REFERENCE_MATRIX_PATH, 'rb') as f:
    ref_df = pickle.load(f)

ref_slide_ids = [str(sid) for sid in ref_df.index.tolist()]
ref_matrix = ref_df.values

print(f"Reference database: {len(ref_slide_ids)} slides")
print(f"Embedding dimension: {ref_matrix.shape[1]}")

# Load metadata for disease labels
metadata = pd.read_csv(METADATA_PATH)
disease_map = dict(zip(metadata['Slide.Scan.File'].astype(str), metadata['Tumor.group']))

# Check query slide's disease
query_disease = disease_map.get(SLIDE_ID, 'Unknown')
print(f"\nQuery slide {SLIDE_ID} disease: {query_disease}")

In [ ]:
# Load slide embedding if not already in memory
try:
    slide_embedding
except NameError:
    slide_emb_path = SLIDE_EMB_DIR / f"{SLIDE_ID}.pkl"
    with open(slide_emb_path, 'rb') as f:
        slide_embedding = pickle.load(f)
    print(f"Loaded slide embedding from: {slide_emb_path}")

# Perform slide search
query = slide_embedding.flatten().reshape(1, -1)

# Compute L1 (cityblock) distances
distances = cdist(query, ref_matrix, metric='cityblock')[0]
sorted_idx = np.argsort(distances)

# Get top-20 results (excluding self)
print("\nTop-20 Similar Slides:")
print("-" * 60)
print(f"{'Rank':<6}{'Slide ID':<12}{'Disease':<15}{'Distance':<12}{'Match'}")
print("-" * 60)

results = []
rank = 0
for idx in sorted_idx:
    sid = ref_slide_ids[idx]
    if sid == SLIDE_ID:
        continue
    rank += 1
    if rank > 20:
        break
    disease = disease_map.get(sid, 'Unknown')
    match = "*" if disease == query_disease else ""
    print(f"{rank:<6}{sid:<12}{disease:<15}{distances[idx]:<12.2f}{match}")
    results.append({
        'query_slide_id': SLIDE_ID,
        'hit_slide_id': sid,
        'distance': distances[idx],
        'rank': rank,
        'hit_disease': disease
    })

# Summary
same_disease_count = sum(1 for r in results if r['hit_disease'] == query_disease)
print(f"\nRetrieval rate ({query_disease}): {same_disease_count}/20 = {same_disease_count/20*100:.0f}%")

In [ ]:
# Save slide search results
slide_search_df = pd.DataFrame(results)
slide_search_path = SEARCH_RESULTS_DIR / f"{SLIDE_ID}_slide_search.csv"
slide_search_df.to_csv(slide_search_path, index=False)
print(f"Saved slide search results to: {slide_search_path}")
slide_search_df.head(10)

### 4.1 Slide Search Visualization

Visualize the slide search results - disease distribution and distance patterns.

In [ ]:
import matplotlib.pyplot as plt

# Disease distribution in top-20 results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Disease distribution in slide search results
ax1 = axes[0]
disease_counts = slide_search_df['hit_disease'].value_counts()
colors = ['#2ecc71' if d == query_disease else '#3498db' for d in disease_counts.index]
ax1.bar(disease_counts.index, disease_counts.values, color=colors)
ax1.set_xlabel('Disease Type')
ax1.set_ylabel('Count')
ax1.set_title(f'Slide Search Results\n(Query: {SLIDE_ID}, Disease: {query_disease})')
ax1.tick_params(axis='x', rotation=45)

# Plot 2: Distance distribution
ax2 = axes[1]
same_disease = slide_search_df[slide_search_df['hit_disease'] == query_disease]['distance']
diff_disease = slide_search_df[slide_search_df['hit_disease'] != query_disease]['distance']

if len(same_disease) > 0:
    ax2.hist(same_disease, bins=10, alpha=0.7, label=f'Same ({query_disease})', color='#2ecc71')
if len(diff_disease) > 0:
    ax2.hist(diff_disease, bins=10, alpha=0.7, label='Different', color='#e74c3c')
ax2.set_xlabel('L1 Distance')
ax2.set_ylabel('Count')
ax2.set_title('Distance Distribution by Disease Match')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 5. ROI Search

Find similar tissue regions (patches) across the reference database.

This is useful for:
- Finding morphologically similar regions in other slides
- Identifying diagnostic patterns
- Quality control and annotation transfer

In [ ]:
from pepath.search import ROISearchEngine

# Initialize ROI search engine
roi_engine = ROISearchEngine(
    top_k=5,  # Top-5 similar patches per query patch
    verbose=True
)

# Load reference patch embeddings
print("Loading reference patch embeddings...")
roi_engine.load_reference_database(
    embeddings_dir=str(REFERENCE_PATCHES_DIR)
)

print(f"Loaded {len(roi_engine.reference_embeddings)} slides for ROI search")

In [ ]:
# Load patch embeddings if not already in memory
# (In case you're running this cell independently)
try:
    patch_embeddings
except NameError:
    patch_emb_path = PATCH_EMB_DIR / "tmp" / f"{SLIDE_ID}.pkl"
    with open(patch_emb_path, 'rb') as f:
        patch_embeddings, mosaic = pickle.load(f)
    print(f"Loaded patch embeddings from: {patch_emb_path}")

# Search for similar patches (using first 10 patches as demo)
n_query_patches = min(10, len(mosaic))  # Limit for demo speed
query_coords = np.array([m['wsi_loc'] for m in mosaic])

print(f"Searching for {n_query_patches} query patches...")
roi_results = roi_engine.search(
    query_embeddings=patch_embeddings,
    query_slide_id=SLIDE_ID,
    query_coords=query_coords,
    patch_indices=list(range(n_query_patches))  # First N patches
)

In [ ]:
# Display ROI search results
print("\nROI Search Results (Top-5 per patch):")
print("-" * 80)

for patch_idx, patch_results in enumerate(roi_results[:5]):  # Show first 5 patches
    query_loc = query_coords[patch_idx]
    print(f"\nQuery Patch {patch_idx} @ ({query_loc[0]}, {query_loc[1]}):")
    for result in patch_results[:3]:  # Top-3 matches
        hit_disease = disease_map.get(result.hit_slide_id, 'Unknown')
        loc_str = f"({result.hit_location[0]}, {result.hit_location[1]})" if result.hit_location else "N/A"
        print(f"  Rank {result.rank}: Slide {result.hit_slide_id} ({hit_disease}) @ {loc_str}, dist={result.distance:.2f}")

In [ ]:
# Save ROI search results
roi_df = roi_engine.results_to_dataframe(roi_results)

# Add disease labels
roi_df['hit_disease'] = roi_df['hit_slide_id'].map(disease_map)

roi_search_path = SEARCH_RESULTS_DIR / f"{SLIDE_ID}_roi_search.csv"
roi_df.to_csv(roi_search_path, index=False)
print(f"Saved ROI search results to: {roi_search_path}")
print(f"Total results: {len(roi_df)} rows")

roi_df.head(10)

### 5.1 ROI Heatmap Visualization

Generate a heatmap showing the similarity of each patch in a matched slide to a query patch.
The heatmap is overlaid on the slide thumbnail for context.

In [ ]:
from pepath.visualization import generate_roi_search_heatmap
from PIL import Image

# Select a query patch and its top hit for heatmap generation
query_patch_idx = 0  # First query patch
top_hit = roi_results[query_patch_idx][0]  # Top-1 match

print(f"Query: Slide {SLIDE_ID}, Patch {query_patch_idx}")
print(f"Hit: Slide {top_hit.hit_slide_id}, Location {top_hit.hit_location}")
print(f"Distance: {top_hit.distance:.2f}")

# Generate heatmap with thumbnail overlay
result_paths = generate_roi_search_heatmap(
    embeddings_dir=str(REFERENCE_PATCHES_DIR),
    query_slide_id=SLIDE_ID,
    query_patch_idx=query_patch_idx,
    hit_slide_id=top_hit.hit_slide_id,
    output_dir=str(HEATMAP_DIR),
    mosaic_path=str(MOSAIC_METADATA_PATH),
    thumbnails_dir=str(THUMBNAILS_DIR),
    dimensions_csv=str(DIMENSIONS_CSV),  # For accurate coordinate mapping
    patch_size=512,
    tile_size=16,
    overlay_alpha=120
)

print(f"\nGenerated heatmap: {result_paths.get('heatmap')}")
print(f"Generated overlay: {result_paths.get('overlay')}")

In [ ]:
# Display the heatmap overlay
if 'overlay' in result_paths:
    overlay_img = Image.open(result_paths['overlay'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Standalone heatmap
    heatmap_img = Image.open(result_paths['heatmap'])
    axes[0].imshow(heatmap_img)
    axes[0].set_title(f'Similarity Heatmap\n(Query: {SLIDE_ID} → Hit: {top_hit.hit_slide_id})')
    axes[0].axis('off')
    
    # Overlay on thumbnail
    axes[1].imshow(overlay_img)
    axes[1].set_title(f'Heatmap Overlay on Thumbnail\n(Warmer colors = More similar)')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nColor scale: Red (most similar) → Violet (least similar)")
else:
    print("Overlay not generated (thumbnail may be missing)")

---
## Summary

This notebook demonstrated the complete PePath pipeline:

1. **Mosaic Generation** - Extracted tissue patches from WSI
2. **Patch Embedding** - Generated Virchow2 embeddings (first 1280 features used)
3. **Slide Embedding** - Aggregated to slide-level using SAMPLER/percentiles (10 x 1280 = 12800-dim)
4. **Slide Search** - Found morphologically similar slides with visualization
5. **ROI Search** - Found similar tissue regions with heatmap overlays

### Output Files
- `output/mosaic/{slide_id}/` - Mosaic patches and metadata
- `output/patch_embeddings/{slide_id}.pkl` - Patch embeddings
- `output/slide_embeddings/{slide_id}.pkl` - Slide embedding
- `output/search_results/{slide_id}_slide_search.csv` - Slide search results
- `output/search_results/{slide_id}_roi_search.csv` - ROI search results
- `output/heatmaps/heatmap_*.png` - ROI similarity heatmaps
- `output/heatmaps/heatmap_*_overlay.png` - Heatmaps overlaid on thumbnails
- `output/ot_heatmaps/ot_overlay_combined_*.png` - OT-based slide comparison heatmaps (via script)

### Key Data Files
- `data/slide_dimensions.csv` - Slide dimensions for accurate heatmap coordinate mapping
- `data/mosaic_metadata.pkl` - Mosaic cluster locations for full tissue coverage
- `data/thumbnails/` - Slide thumbnails for overlay visualization

### Next Steps
- Process more slides by changing `SLIDE_PATH`
- Try different embedding methods: `bob`, `ot`
- Generate OT-based slide comparison heatmaps using `scripts/test_ot_heatmap.py`
  - Supports side-by-side visualization of two slides with overlays on thumbnails

In [ ]:
print("Demo complete!")
print(f"\nGenerated files:")
print(f"  - Slide search: {slide_search_path}")
print(f"  - ROI search: {roi_search_path}")
if 'result_paths' in dir() and result_paths:
    print(f"  - Heatmap: {result_paths.get('heatmap', 'N/A')}")
    print(f"  - Heatmap overlay: {result_paths.get('overlay', 'N/A')}")